# FEVER Small-Model CALE Visualization

This notebook is tailored for the full FEVER internal evaluation report generated by `run_small_models_all_datasets.sh`.

Default input:

`outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_report.json`

It creates paper-facing figures for model comparison, evaluator comparison, FEVER label behavior, CALE construct subscores, and qualitative disagreement examples.

In [ ]:
from pathlib import Path
import json
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 10,
})

RESULTS_PATH = Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_report.json")
OUTDIR = Path("figures") / RESULTS_PATH.stem
OUTDIR.mkdir(parents=True, exist_ok=True)

raw_text = RESULTS_PATH.read_text(encoding="utf-8")
nonempty_lines = [line for line in raw_text.splitlines() if line.strip()]
if len(nonempty_lines) > 1 and all(line.lstrip().startswith("{") for line in nonempty_lines[:2]):
    raise ValueError(
        "RESULTS_PATH points to a JSONL response file, not the report JSON from experiment.py. "
        "Use the *_eval_report.json file printed by run_pipeline.sh."
    )

report = json.loads(raw_text)
required = {"run_config", "dataset_summary", "metrics", "metrics_by_model"}
missing = required - set(report)
if missing:
    raise ValueError(f"Report is missing required keys: {sorted(missing)}")

pred_df = pd.DataFrame(report.get("predictions", []))
print(f"Loaded report: {RESULTS_PATH}")
print(f"Figures/tables will be written to: {OUTDIR}")
print(f"Prediction rows: {len(pred_df):,}")
report.keys()

## Run Summary

In [ ]:
run_config = report["run_config"]
dataset_summary = report["dataset_summary"]

summary_rows = [
    ("dataset_path", run_config.get("dataset_path")),
    ("judge", run_config.get("judge")),
    ("repeats", run_config.get("repeats")),
    ("stress", run_config.get("stress")),
    ("summary_only", run_config.get("summary_only")),
    ("n_items", dataset_summary.get("n_items")),
    ("target_models", ", ".join(run_config.get("target_models", []))),
    ("variants", ", ".join(run_config.get("variants", []))),
]
run_summary_df = pd.DataFrame(summary_rows, columns=["field", "value"])
display(run_summary_df)

model_counts = pd.Series(dataset_summary.get("by_model_name", {}), name="n").sort_values(ascending=False)
display(model_counts.to_frame())

fig, ax = plt.subplots(figsize=(7, 3.2))
model_counts.plot(kind="bar", ax=ax, color="#3B82F6")
ax.set_title("FEVER Evaluation Rows by Target Model")
ax.set_ylabel("Rows")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(OUTDIR / "dataset_rows_by_model.png")

## Metric Tables

In [ ]:
def flatten_variant_metrics(metric_dict, group_name=None):
    rows = []
    for variant, values in metric_dict.items():
        row = {"variant": variant}
        if group_name is not None:
            row["group"] = group_name
        row.update(values)
        rows.append(row)
    return pd.DataFrame(rows)

def flatten_grouped_metrics(grouped_key):
    rows = []
    for group, variant_metrics in report.get(grouped_key, {}).items():
        df = flatten_variant_metrics(variant_metrics, group_name=group)
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

metrics_df = flatten_variant_metrics(report["metrics"])
model_metrics_df = flatten_grouped_metrics("metrics_by_model")

paper_metrics = [
    "mean_uncertainty",
    "misinformation_detection_rate",
    "framing_resistance_rate",
    "source_faithfulness_rate",
    "overclaim_rate_on_nei",
    "accuracy",
    "macro_f1",
    "checklist_f1",
]

display(metrics_df[[c for c in ["variant", "n", *paper_metrics] if c in metrics_df.columns]])
display(model_metrics_df[[c for c in ["group", "variant", "n", *paper_metrics] if c in model_metrics_df.columns]].head(20))

## Evaluator Variant Comparison

In [ ]:
def nice_ylim(values, metric, min_span=0.04):
    values = pd.Series(values).dropna().astype(float)
    if values.empty:
        return (0, 1)
    ymin, ymax = values.min(), values.max()
    span = max(ymax - ymin, min_span)
    pad = span * 0.35
    lower = ymin - pad
    upper = ymax + pad
    if metric != "mean_uncertainty":
        lower = max(0, lower)
        upper = min(1, upper)
    if upper <= lower:
        upper = lower + min_span
    return lower, upper

def plot_overall_metric(metric):
    if metric not in metrics_df.columns or metrics_df[metric].dropna().empty:
        print(f"Skipping {metric}: no non-null values.")
        return
    df = metrics_df[["variant", metric]].dropna().copy()
    df = df.sort_values(metric, ascending=False)
    colors = ["#0F766E" if value >= df[metric].mean() else "#94A3B8" for value in df[metric]]

    fig, ax = plt.subplots(figsize=(10.5, 4.2))
    bars = ax.bar(df["variant"], df[metric], color=colors)
    ax.axhline(df[metric].mean(), color="#334155", linestyle="--", linewidth=1, label=f"mean={df[metric].mean():.3f}")
    ax.set_title(f"Evaluator Variant Comparison: {metric} (zoomed scale)")
    ax.set_ylabel(metric)
    ax.set_xlabel("")
    ax.set_ylim(*nice_ylim(df[metric], metric))
    ax.tick_params(axis="x", rotation=25)
    ax.legend(frameon=False, loc="best")
    for bar, value in zip(bars, df[metric]):
        ax.annotate(
            f"{value:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, value),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    fig.tight_layout()
    fig.savefig(OUTDIR / f"overall_{metric}_zoomed.png")

    delta = df.assign(delta=df[metric] - df[metric].mean()).sort_values("delta")
    fig, ax = plt.subplots(figsize=(9, 3.8))
    delta_colors = ["#DC2626" if value < 0 else "#2563EB" for value in delta["delta"]]
    bars = ax.barh(delta["variant"], delta["delta"], color=delta_colors)
    ax.axvline(0, color="#334155", linewidth=1)
    ax.set_title(f"Difference from Evaluator Mean: {metric}")
    ax.set_xlabel(f"{metric} - mean")
    for bar, value in zip(bars, delta["delta"]):
        x_offset = 4 if value >= 0 else -4
        ha = "left" if value >= 0 else "right"
        ax.annotate(f"{value:+.3f}", xy=(value, bar.get_y() + bar.get_height() / 2), xytext=(x_offset, 0), textcoords="offset points", va="center", ha=ha, fontsize=9)
    fig.tight_layout()
    fig.savefig(OUTDIR / f"overall_{metric}_delta.png")

for metric in ["source_faithfulness_rate", "misinformation_detection_rate", "framing_resistance_rate", "overclaim_rate_on_nei", "mean_uncertainty"]:
    plot_overall_metric(metric)

## Model x Evaluator Heatmaps

In [ ]:
def heatmap_from_pivot(pivot, title, filename, vmin=None, vmax=None, cmap="viridis"):
    if pivot.empty:
        print(f"Skipping empty heatmap: {title}")
        return
    fig, ax = plt.subplots(figsize=(max(8, 2.8 * len(pivot.columns)), max(4.2, 0.55 * len(pivot))))
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=20, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            label = "" if pd.isna(val) else f"{val:.2f}"
            ax.text(j, i, label, ha="center", va="center", color="white" if not pd.isna(val) and val < 0.55 else "black")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    fig.tight_layout()
    fig.savefig(OUTDIR / filename)

for metric in ["source_faithfulness_rate", "misinformation_detection_rate", "framing_resistance_rate", "overclaim_rate_on_nei", "mean_uncertainty"]:
    if model_metrics_df.empty or metric not in model_metrics_df.columns or model_metrics_df[metric].dropna().empty:
        print(f"Skipping model heatmap for {metric}.")
        continue
    pivot = model_metrics_df.pivot_table(index="variant", columns="group", values=metric, aggfunc="mean")
    heatmap_from_pivot(pivot, f"Model x Evaluator: {metric}", f"model_variant_{metric}.png", vmin=0, vmax=1)

## Score and Label Distributions

In [ ]:
if pred_df.empty:
    raise ValueError("This report does not include raw predictions. Re-run experiment.py without --summary-only for row-level plots.")

pred_df["model_short"] = pred_df["model_name"].str.replace("Qwen/", "", regex=False).str.replace("meta-llama/", "", regex=False)

score_summary = (
    pred_df.groupby(["model_short", "variant"], dropna=False)["score"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values(["variant", "model_short"])
)
display(score_summary.head(20))

selected_variant = "full_attack_aware_cale" if "full_attack_aware_cale" in pred_df["variant"].unique() else pred_df["variant"].iloc[0]
sub = pred_df[pred_df["variant"] == selected_variant].copy()

fig, ax = plt.subplots(figsize=(8, 4))
sub.boxplot(column="score", by="model_short", ax=ax, grid=False, rot=15)
ax.set_title(f"Final CALE Score by Model ({selected_variant})")
ax.set_xlabel("")
ax.set_ylabel("score")
fig.suptitle("")
fig.tight_layout()
fig.savefig(OUTDIR / "score_distribution_by_model.png")

label_counts = sub.groupby(["model_short", "label"]).size().rename("n").reset_index()
label_counts["share"] = label_counts["n"] / label_counts.groupby("model_short")["n"].transform("sum")
pivot = label_counts.pivot(index="model_short", columns="label", values="share").fillna(0)
ax = pivot.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="Set2")
ax.set_title(f"Predicted Label Distribution by Model ({selected_variant})")
ax.set_ylabel("Share")
ax.set_xlabel("")
ax.legend(title="CALE label", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTDIR / "label_distribution_by_model.png")

## FEVER Reference-Label Behavior

This section is useful for factuality analysis. In particular, `NOT ENOUGH INFO` is where overclaiming behavior is most visible.

In [ ]:
if "reference_label" not in pred_df.columns or pred_df["reference_label"].dropna().empty:
    print("No FEVER reference_label found in predictions.")
else:
    fever = pred_df[pred_df["variant"] == selected_variant].copy()
    ref_label_table = fever.groupby(["model_short", "reference_label", "label"]).size().rename("n").reset_index()
    ref_label_table["share"] = ref_label_table["n"] / ref_label_table.groupby(["model_short", "reference_label"])["n"].transform("sum")
    display(ref_label_table.head(30))

    for model_name, model_df in fever.groupby("model_short"):
        model_counts = model_df.groupby(["reference_label", "label"]).size().rename("n").reset_index()
        model_counts["share"] = model_counts["n"] / model_counts.groupby("reference_label")["n"].transform("sum")
        pivot = model_counts.pivot(index="reference_label", columns="label", values="share").fillna(0)
        ax = pivot.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="Set2")
        ax.set_title(f"CALE Label Distribution by FEVER Label: {model_name}")
        ax.set_ylabel("Share")
        ax.set_xlabel("FEVER reference label")
        ax.legend(title="CALE label", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        safe_model = model_name.replace("/", "_")
        plt.savefig(OUTDIR / f"reference_label_distribution_{safe_model}.png")

    if "NOT ENOUGH INFO" in set(fever["reference_label"].dropna()):
        nei = fever[fever["reference_label"] == "NOT ENOUGH INFO"].copy()
        nei_overclaim = nei.assign(overclaim=nei["label"].isin(["corrected", "partially_corrected"]).astype(float))
        nei_summary = nei_overclaim.groupby("model_short")["overclaim"].mean().sort_values(ascending=False)
        display(nei_summary.to_frame("overclaim_rate_on_nei"))
        fig, ax = plt.subplots(figsize=(7, 3.5))
        nei_summary.plot(kind="bar", ax=ax, color="#DC2626")
        ax.set_title(f"Overclaim Rate on FEVER NEI ({selected_variant})")
        ax.set_ylabel("Overclaim rate")
        ax.set_xlabel("")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=15)
        fig.tight_layout()
        fig.savefig(OUTDIR / "nei_overclaim_rate_by_model.png")

## CALE Construct Subscores

This is the most CALE-specific view: it shows whether the model/evaluator differences come from misinformation detection, framing resistance, source faithfulness, or other rubric dimensions.

In [ ]:
subscore_rows = []
for row in pred_df.to_dict("records"):
    subscores = row.get("subscores") or {}
    if not isinstance(subscores, dict):
        continue
    for dimension, value in subscores.items():
        subscore_rows.append({
            "id": row.get("id"),
            "model_short": row.get("model_short"),
            "variant": row.get("variant"),
            "reference_label": row.get("reference_label"),
            "dimension": dimension,
            "score": value,
        })

subscore_df = pd.DataFrame(subscore_rows)
display(subscore_df.head())

if subscore_df.empty:
    print("No CALE subscores found. Baseline-only reports will not have construct-level plots.")
else:
    construct = subscore_df[subscore_df["variant"] == selected_variant]
    pivot = construct.pivot_table(index="dimension", columns="model_short", values="score", aggfunc="mean")
    heatmap_from_pivot(pivot, f"Mean CALE Construct Subscores ({selected_variant})", "construct_subscores_by_model.png", vmin=0, vmax=1)

    if "Source Faithfulness" in set(subscore_df["dimension"]):
        sf = subscore_df[(subscore_df["dimension"] == "Source Faithfulness") & (subscore_df["variant"] == selected_variant)]
        sf_summary = sf.groupby("model_short")["score"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
        display(sf_summary)
        fig, ax = plt.subplots(figsize=(7, 3.5))
        sf_summary["mean"].plot(kind="bar", ax=ax, color="#2563EB")
        ax.set_title(f"Source Faithfulness by Model ({selected_variant})")
        ax.set_ylabel("Mean Source Faithfulness")
        ax.set_xlabel("")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=15)
        fig.tight_layout()
        fig.savefig(OUTDIR / "source_faithfulness_by_model.png")

## Model Disagreement and Qualitative Examples

In [ ]:
def show_text(value, width=120):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return textwrap.shorten(str(value).replace("\n", " "), width=width, placeholder=" ...")

chosen = pred_df[pred_df["variant"] == selected_variant].copy()
if chosen["model_short"].nunique() < 2:
    print("Need at least two target models for disagreement analysis.")
else:
    label_pivot = chosen.pivot_table(index="id", columns="model_short", values="label", aggfunc="first")
    score_pivot = chosen.pivot_table(index="id", columns="model_short", values="score", aggfunc="first")
    complete = label_pivot.dropna()
    disagreement_mask = complete.nunique(axis=1) > 1
    disagreement_ids = complete.index[disagreement_mask]
    print(f"Disagreement rows for {selected_variant}: {len(disagreement_ids):,} / {len(complete):,}")

    fig, ax = plt.subplots(figsize=(5.8, 3.3))
    pd.Series({"agree": len(complete) - len(disagreement_ids), "disagree": len(disagreement_ids)}).plot(kind="bar", ax=ax, color=["#16A34A", "#F97316"])
    ax.set_title(f"Model Label Agreement ({selected_variant})")
    ax.set_ylabel("Items")
    ax.set_xlabel("")
    fig.tight_layout()
    fig.savefig(OUTDIR / "model_label_agreement.png")

    examples = chosen[chosen["id"].isin(disagreement_ids)].copy()
    examples = examples.sort_values(["id", "model_short"])
    cols = [c for c in ["id", "model_short", "reference_label", "label", "score", "uncertainty"] if c in examples.columns]
    display(examples[cols].head(20))

    if not score_pivot.empty:
        score_gap = score_pivot.max(axis=1) - score_pivot.min(axis=1)
        top_gap_ids = score_gap.sort_values(ascending=False).head(10).index
        top_gap = chosen[chosen["id"].isin(top_gap_ids)].copy().sort_values(["id", "model_short"])
        display(top_gap[[c for c in ["id", "model_short", "reference_label", "label", "score", "uncertainty"] if c in top_gap.columns]])

## Export Paper Tables

In [ ]:
metrics_df.to_csv(OUTDIR / "overall_metrics.csv", index=False)
model_metrics_df.to_csv(OUTDIR / "model_metrics.csv", index=False)
if not pred_df.empty:
    score_summary.to_csv(OUTDIR / "score_summary_by_model_variant.csv", index=False)
if "subscore_df" in globals() and not subscore_df.empty:
    subscore_df.groupby(["model_short", "variant", "dimension"], dropna=False)["score"].agg(["mean", "std", "count"]).reset_index().to_csv(
        OUTDIR / "construct_subscores_summary.csv", index=False
    )

print("Saved figures and tables to:", OUTDIR)